# Deep learning experiments

Develop a temporal GRU, geographical and multi-graph extensions, and an ST-MGCN-inspired model. The standalone Avon workflow is in `../spatiotemporal_forecasting.py`.

## Imports


In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import geopandas as gpd
import data_tools as data
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt


## Load zone geometry and demand


In [ ]:
df_zones = gpd.read_file(
    "/Users/sai-z/Desktop/nyc-taxi-demand-forecasting/taxi_zones/taxi_zones.shp"
)
df = pd.read_parquet(
    "/Users/sai-z/Desktop/nyc-taxi-demand-forecasting/data/DL_Dataset.parquet"
)


## Geographical graph


In [ ]:
# Neighborhood
adjacency = np.zeros((263, 263))
for i in range(263):
    boundary = df_zones.geometry.iloc[i].boundary
    shared_parts = df_zones.boundary.intersection(boundary)
    mask = ~shared_parts.is_empty
    neighbors = df_zones.loc[mask].drop(index=df_zones.index[i])
    for j in neighbors.LocationID:
        adjacency[i][j - 1] += 1
adjacency = adjacency.astype(int)


## Functional-similarity graph


In [ ]:
# Functionnal similarity
# One row per zone, ordered by zone ID
poi = (
    df.groupby("PULocationID")[["restaurants", "hotels"]]
    .first()
    .reindex(range(1, 264))
    .to_numpy(dtype=float)
)

norms = np.linalg.norm(poi, axis=1, keepdims=True)

# Normalize nonzero vectors; leave zero vectors as zeros
poi_normalized = np.divide(
    poi,
    norms,
    out=np.zeros_like(poi),
    where=norms > 0,
)

functionnality = poi_normalized @ poi_normalized.T


## Subway connectivity graph


In [ ]:
df_subway = pd.read_csv(
    "/Users/sai-z/Desktop/nyc-taxi-demand-forecasting/MTA_Subway_Entrances_and_Exits__2024_20261021.csv"
)


In [ ]:
# Transportation connectivity
# Rows/columns are ordered by LocationID 1,...,263; diagonal is zero.
# This is shared-route connectivity, not consecutive-stop or transfer connectivity.
from pathlib import Path

project_dir = Path("/Users/sai-z/Desktop/nyc-taxi-demand-forecasting")
zone_ids = pd.Index(range(1, 264), name="LocationID")

# Coordinates use decimal commas in this CSV.
subway = df_subway.copy()
for column in ["Entrance Latitude", "Entrance Longitude"]:
    subway[column] = pd.to_numeric(
        subway[column].astype(str).str.replace(",", ".", regex=False),
        errors="coerce",
    )
if subway[["Entrance Latitude", "Entrance Longitude"]].isna().any().any():
    raise ValueError("Missing or invalid subway entrance coordinates")

entrances = gpd.GeoDataFrame(
    subway,
    geometry=gpd.points_from_xy(
        subway["Entrance Longitude"], subway["Entrance Latitude"]
    ),
    crs="EPSG:4326",
).to_crs(df_zones.crs)

# Union geometry parts belonging to the same zone before the spatial join.
zones = df_zones[["LocationID", "geometry"]].dissolve(by="LocationID").reset_index()
assigned = gpd.sjoin(entrances, zones, how="left", predicate="intersects")
unmatched = assigned.loc[assigned["LocationID"].isna()].copy()
ambiguous = assigned.loc[assigned.index.duplicated(keep=False)].copy()
if not ambiguous.empty:
    raise ValueError("Some entrances intersect multiple zones; inspect ambiguous")

# Multiple entrances/stations on the same route in one zone count only once.
zone_routes = assigned.dropna(subset=["LocationID", "Daytime Routes"]).copy()
zone_routes["LocationID"] = zone_routes["LocationID"].astype(int)
zone_routes["route"] = zone_routes["Daytime Routes"].str.split()
zone_routes = (
    zone_routes.explode("route")
    .drop_duplicates(["LocationID", "route"])
    .reset_index(drop=True)
)
route_presence = pd.crosstab(zone_routes["LocationID"], zone_routes["route"])
route_presence = route_presence.reindex(zone_ids, fill_value=0).gt(0).astype("int32")

# B @ B.T counts shared routes; threshold to obtain binary connectivity.
B = route_presence.to_numpy()
transportation = (B @ B.T > 0).astype(int)
np.fill_diagonal(transportation, 0)

connectivity = np.maximum(0, transportation - adjacency).astype(int)


## Hourly demand matrix


In [ ]:
timestamps = pd.to_datetime(df["tpep_pickup_datetime"])

df["hour_of_year"] = (timestamps.dt.dayofyear - 1) * 24 + timestamps.dt.hour + 1


In [ ]:
demand = (
    df.groupby(["hour_of_year", "PULocationID"])["trip_count"]
    .sum()
    .unstack("PULocationID")
)
D = demand.to_numpy()


## Chronological split and training-only scaling


In [ ]:
X = df.copy()


In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test, train, val, test = data.split(X)


In [ ]:
mean_train = np.mean(y_train)
std_train = np.std(y_train)
D_scaled = (D - mean_train) / std_train


## Sliding windows and PyTorch batches


In [ ]:
Input = []
target = []
for i in range(8784 - 24):
    Input.append(D_scaled[i : 24 + i, :])
    target.append(D_scaled[24 + i])
Input = np.array(Input)
target = np.array(target)


In [ ]:
training = torch.tensor(Input[:5832], dtype=torch.float32)
target_train = torch.tensor(target[:5832], dtype=torch.float32)
validation = torch.tensor(Input[5832 : 5832 + 720], dtype=torch.float32)
target_val = torch.tensor(target[5832 : 5832 + 720], dtype=torch.float32)
testing = torch.tensor(Input[5832 + 720 :], dtype=torch.float32)
target_test = torch.tensor(target[5832 + 720 :], dtype=torch.float32)


In [ ]:
train_input_target = TensorDataset(training, target_train)
val_input_target = TensorDataset(validation, target_val)
test_input_target = TensorDataset(testing, target_test)


In [ ]:
train_loader = DataLoader(train_input_target, batch_size=64, shuffle=True)
val_loader = DataLoader(val_input_target, batch_size=64, shuffle=False)
test_loader = DataLoader(test_input_target, batch_size=64, shuffle=False)


In [ ]:
X_batch, y_batch = next(iter(train_loader))

print(X_batch.shape)
print(y_batch.shape)


## Temporal GRU baseline


In [ ]:
# GRU
class GRU(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.GRU = nn.GRU(
            input_size=input_dim, hidden_size=hidden_dim, batch_first=True
        )
        self.Linear = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        output, x = self.GRU(x)
        x = x[-1]
        x = self.Linear(x)
        return x


In [ ]:
GRU_model = GRU(263, 64, 263)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(GRU_model.parameters(), lr=0.001)
num_epochs = 30
loss_epoch = []
rmse_epoch = []
loss_epoch_val = []
rmse_epoch_val = []

for epoch in range(num_epochs):
    GRU_model.train()
    loss_tot = 0
    for X, y in train_loader:
        optimizer.zero_grad()
        prediction = GRU_model(X)
        loss = criterion(prediction, y)
        loss_tot += loss.item() * X.size(0)
        loss.backward()
        optimizer.step()
    loss_epoch.append(loss_tot / len(train_loader.dataset))
    rmse_epoch.append(np.sqrt(loss_tot / len(train_loader.dataset)) * std_train)

    GRU_model.eval()
    loss_tot_val = 0
    with torch.no_grad():
        for X, y in val_loader:
            prediction = GRU_model(X)
            loss_tot_val += criterion(prediction, y).item() * X.size(0)

    val_loss = loss_tot_val / len(val_loader.dataset)
    loss_epoch_val.append(val_loss)
    rmse_epoch_val.append(np.sqrt(val_loss) * std_train)


## Baseline learning curves


In [ ]:
epochs = [n for n in range(num_epochs)]
plt.plot(epochs, rmse_epoch)
plt.plot(epochs, rmse_epoch_val)
plt.xlabel("Epoch")
plt.ylabel("RMSE")
plt.show()


## Graph normalisation


In [ ]:
adjacency_sc = adjacency + np.eye(263)
degree_matrix = np.zeros((263, 263))
for i in range(len(degree_matrix)):
    degree_matrix[i][i] = 1 / np.sqrt(np.sum(adjacency_sc[i, :]))

adjacency_normalized = degree_matrix @ adjacency_sc @ degree_matrix


In [ ]:
functionnality_sc = functionnality.copy()
np.fill_diagonal(functionnality_sc, 1)
degree_matrix_1 = np.zeros((263, 263))
for i in range(len(degree_matrix)):
    degree_matrix_1[i][i] = 1 / np.sqrt(np.sum(functionnality_sc[i, :]))
functionnality_normalized = degree_matrix_1 @ functionnality_sc @ degree_matrix_1


In [ ]:
degree_matrix_2 = np.zeros((263, 263))
for i in range(len(degree_matrix_2)):
    if np.sum(connectivity[i, :]) != 0:
        degree_matrix_2[i][i] = 1 / np.sqrt(np.sum(connectivity[i, :]))
connectivity_normalized = degree_matrix_2 @ connectivity @ degree_matrix_2


## Multi-graph GRU architecture


In [ ]:
class GRU_Geo(nn.Module):
    def __init__(
        self, adjacency_normalized, functionnality_normalized, connectivity_normalized
    ):
        super().__init__()
        self.register_buffer(
            "adjacency", torch.as_tensor(adjacency_normalized, dtype=torch.float32)
        )
        self.register_buffer(
            "functionnality",
            torch.as_tensor(functionnality_normalized, dtype=torch.float32),
        )
        self.register_buffer(
            "connectivity",
            torch.as_tensor(connectivity_normalized, dtype=torch.float32),
        )

        self.gru = nn.GRU(input_size=4, hidden_size=64, batch_first=True)
        self.output = nn.Linear(64, 1)
        self.spatial = nn.Linear(8, 1)

    def forward(self, x):
        batch_size, hours, zones = x.shape
        x = x.unsqueeze(-1)
        x = torch.relu(self.spatial(x))
        x_adj = torch.einsum("ij,btjf->btif", self.adjacency, x)
        x_fun = torch.einsum("ij,btjf->btif", self.functionnality, x)
        x_conn = torch.einsum("ij,btjf->btif", self.connectivity, x)
        x = torch.cat([x, x_adj, x_fun, x_conn], dim=-1)
        x = x.permute(0, 2, 1, 3)
        x = x.reshape(batch_size * zones, hours, 4)
        _, hidden = self.gru(x)
        x = self.output(hidden[-1])
        return x.reshape(batch_size, zones)


## Archived training example

The following cell retains the existing disabled training block.


In [ ]:
'''
GRU_GEO = GRU_Geo(adjacency_normalized)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(GRU_GEO.parameters(), lr=0.001)
num_epochs=3
loss_epoch=[]
rmse_epoch=[]
loss_epoch_val=[]
rmse_epoch_val=[]

for epoch in range(num_epochs):
    GRU_GEO.train()
    loss_tot = 0
    for X,y in train_loader:
        optimizer.zero_grad()
        prediction = GRU_GEO(X)
        loss = criterion(prediction,y)
        loss_tot+=loss.item()*X.size(0)
        loss.backward()
        optimizer.step()  
    loss_epoch.append(loss_tot/len(train_loader.dataset))  
    rmse_epoch.append(np.sqrt(loss_tot/len(train_loader.dataset))*std_train)
    
    GRU_GEO.eval()
    loss_tot_val=0
    with torch.no_grad():
        for X,y in val_loader:
            prediction = GRU_GEO(X)
            loss_tot_val += criterion(prediction,y).item()*X.size(0)
        
    val_loss = loss_tot_val/len(val_loader.dataset)
    loss_epoch_val.append(val_loss)
    rmse_epoch_val.append(np.sqrt(val_loss)*std_train)
    
    
epochs = [n for n in range(num_epochs)]
plt.plot(epochs,rmse_epoch)
plt.plot(epochs,rmse_epoch_val)
plt.xlabel("Epoch")
plt.ylabel("RMSE")
plt.show()
'''

## ST-MGCN-inspired model


## Contextual recurrent encoding and Chebyshev graph layers


In [ ]:
# Laplacian
L_adj = np.eye(263) - adjacency_normalized
L_fun = np.eye(263) - functionnality_normalized
L_conn = np.eye(263) - connectivity_normalized


class CGRNN(nn.Module):
    def __init__(self, L):
        super().__init__()
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()
        self.RNN = nn.RNN(input_size=1, hidden_size=64, batch_first=True)
        self.K = 1
        self.alpha = nn.Parameter(torch.randn(self.K + 1))
        self.attention1 = nn.Linear(2, 8)
        self.attention2 = nn.Linear(8, 1)
        self.register_buffer("L", torch.as_tensor(L, dtype=torch.float32))

    def forward(self, x):
        B, T, N = x.shape
        x_copy = x
        alpha0, alpha1 = self.alpha
        f = self.sigmoid(alpha0 * x + alpha1 * (x_copy @ self.L))
        x_copy = torch.stack([x_copy, f], dim=-1)
        x_copy = x_copy.mean(dim=2)
        x_copy = self.relu(self.attention1(x_copy))
        x_copy = self.sigmoid(self.attention2(x_copy))
        x_copy = x * x_copy
        x_copy = x_copy.permute(0, 2, 1)
        x_copy = x_copy.reshape(B * N, T, 1)
        _, hidden = self.RNN(x_copy)
        x_copy = hidden[-1].reshape(B, N, 64)
        return x_copy


# Chebychev
def cheb(k, L):
    L = (2 / np.max(np.linalg.eigvalsh(L))) * L - np.eye(len(L))
    if k == 0:
        return np.eye(len(L))
    elif k == 1:
        return L
    else:
        return 2 * L @ cheb(k - 1, L) - cheb(k - 2, L)


class ST_MGCN_init(nn.Module):
    def __init__(
        self,
        adjacency_normalized,
        functionnality_normalized,
        connectivity_normalized,
        input_dim,
        output_dim,
        L_fun,
        L_adj,
        L_conn,
    ):
        super().__init__()
        self.K = 2
        self.theta_adj = nn.Parameter(torch.randn(self.K + 1))
        self.theta_fun = nn.Parameter(torch.randn(self.K + 1))
        self.theta_conn = nn.Parameter(torch.randn(self.K + 1))
        self.relu = nn.ReLU()
        self.register_buffer(
            "adjacency", torch.as_tensor(adjacency_normalized, dtype=torch.float32)
        )
        self.register_buffer(
            "functionnality",
            torch.as_tensor(functionnality_normalized, dtype=torch.float32),
        )
        self.register_buffer(
            "connectivity",
            torch.as_tensor(connectivity_normalized, dtype=torch.float32),
        )
        self.register_buffer("L_adj", torch.as_tensor(L_adj, dtype=torch.float32))
        self.register_buffer("L_conn", torch.as_tensor(L_conn, dtype=torch.float32))
        self.register_buffer("L_fun", torch.as_tensor(L_fun, dtype=torch.float32))
        self.cgrnn_adj = CGRNN(L_adj)
        self.cgrnn_fun = CGRNN(L_fun)
        self.cgrnn_conn = CGRNN(L_conn)
        self.linear = nn.Linear(input_dim, output_dim)

    def forward(self, x):
        x_adj = self.cgrnn_adj(x)
        x_fun = self.cgrnn_fun(x)
        x_conn = self.cgrnn_conn(x)
        theta0, theta1, theta2 = self.theta_adj
        theta00, theta11, theta22 = self.theta_fun
        theta000, theta111, theta222 = self.theta_conn
        adj = (
            theta0 * torch.eye(263, device=self.L_adj.device)
            + theta1 * self.L_adj
            + theta2 * (self.L_adj @ self.L_adj)
        )
        fun = (
            theta00 * torch.eye(263, device=self.L_fun.device)
            + theta11 * self.L_fun
            + theta22 * (self.L_fun @ self.L_fun)
        )
        conn = (
            theta000 * torch.eye(263, device=self.L_conn.device)
            + theta111 * self.L_conn
            + theta222 * (self.L_conn @ self.L_conn)
        )
        x_adj = self.linear(x_adj)
        x_fun = self.linear(x_fun)
        x_conn = self.linear(x_conn)
        x_adj = adj @ x_adj
        x_fun = fun @ x_fun
        x_conn = conn @ x_conn
        x_next = self.relu(x_adj + x_fun + x_conn)
        return x_next


# Between layers
class ST_MGCN_layer(nn.Module):
    def __init__(
        self,
        adjacency_normalized,
        functionnality_normalized,
        connectivity_normalized,
        input_dim,
        output_dim,
        L_fun,
        L_adj,
        L_conn,
    ):
        super().__init__()
        self.K = 2
        self.theta_adj = nn.Parameter(torch.randn(self.K + 1))
        self.theta_fun = nn.Parameter(torch.randn(self.K + 1))
        self.theta_conn = nn.Parameter(torch.randn(self.K + 1))
        self.relu = nn.ReLU()
        self.register_buffer("L_adj", torch.as_tensor(L_adj, dtype=torch.float32))
        self.register_buffer("L_conn", torch.as_tensor(L_conn, dtype=torch.float32))
        self.register_buffer("L_fun", torch.as_tensor(L_fun, dtype=torch.float32))
        self.linear = nn.Linear(input_dim, output_dim)

    def forward(self, x):
        theta0, theta1, theta2 = self.theta_adj
        theta00, theta11, theta22 = self.theta_fun
        theta000, theta111, theta222 = self.theta_conn
        adj = (
            theta0 * torch.eye(263, device=self.L_adj.device)
            + theta1 * self.L_adj
            + theta2 * (self.L_adj @ self.L_adj)
        )
        fun = (
            theta00 * torch.eye(263, device=self.L_fun.device)
            + theta11 * self.L_fun
            + theta22 * (self.L_fun @ self.L_fun)
        )
        conn = (
            theta000 * torch.eye(263, device=self.L_conn.device)
            + theta111 * self.L_conn
            + theta222 * (self.L_conn @ self.L_conn)
        )
        x = self.linear(x)
        x_next = self.relu((adj + fun + conn) @ x)
        return x_next


class ST_MGCN(nn.Module):
    def __init__(
        self,
        adjacency_normalized,
        functionnality_normalized,
        connectivity_normalized,
        L_fun,
        L_adj,
        L_conn,
    ):
        super().__init__()
        self.stmgcn_i = ST_MGCN_init(
            adjacency_normalized,
            functionnality_normalized,
            connectivity_normalized,
            input_dim=64,
            output_dim=64,
        )
        self.layers = nn.ModuleList(
            [
                ST_MGCN_layer(
                    adjacency_normalized,
                    functionnality_normalized,
                    connectivity_normalized,
                    input_dim=64,
                    output_dim=64,
                    L_fun=L_fun,
                    L_adj=L_adj,
                    L_conn=L_conn,
                )
                for _ in range(2)
            ]
        )
        self.prediction = nn.Linear(64, 1)

    def forward(self, x):
        x = self.stmgcn_i(x)
        for layer in self.layers:
            x = layer(x)
        x = self.prediction(x)
        return x.squeeze(-1)


model = ST_MGCN(
    adjacency_normalized,
    functionnality_normalized,
    connectivity_normalized,
    L_adj=L_adj,
    L_fun=L_fun,
    L_conn=L_conn,
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
num_epochs = 3
loss_epoch = []
rmse_epoch = []
loss_epoch_val = []
rmse_epoch_val = []

best = -np.inf
for epoch in range(num_epochs):
    model.train()
    loss_tot = 0
    for X, y in train_loader:
        X = X.to(device)
        y = y.to(device)
        optimizer.zero_grad()
        prediction = model(X)
        loss = criterion(prediction, y)
        loss_tot += loss.item() * X.size(0)
        loss.backward()
        optimizer.step()
    loss_epoch.append(loss_tot / len(train_loader.dataset))
    rmse_epoch.append(np.sqrt(loss_tot / len(train_loader.dataset)) * std_train)

    model.eval()
    loss_tot_val = 0
    with torch.no_grad():
        for X, y in val_loader:
            X = X.to(device)
            y = y.to(device)
            prediction = model(X)
            loss_tot_val += criterion(prediction, y).item() * X.size(0)

    val_loss = loss_tot_val / len(val_loader.dataset)
    loss_epoch_val.append(val_loss)
    rmse_epoch_val.append(np.sqrt(val_loss) * std_train)
    best = min(best, np.sqrt(val_loss) * std_train)


epochs = [n for n in range(num_epochs)]
print(best)
plt.plot(epochs, rmse_epoch)
plt.plot(epochs, rmse_epoch_val)
plt.xlabel("Epoch")
plt.ylabel("RMSE")
plt.savefig("ST_MGCN_training.png", dpi=150)
